# Experiment Reproduction:

*Multi-agent Deep Reinforcement Learning collaborative Traffic
Signal Control method considering intersection heterogeneity*
- Yiming Bie a
- Yuting Ji a
- Dongfang Ma

In [1]:
# Initialize by cloning the repository from GitHub
import os
repo_url = "https://github.com/IsaacFayle-Waters/MARL-TSC-SUMO-Group.git"
if not os.path.exists('MARL-TSC-SUMO-Group'):
    !git clone {repo_url}
else:
    print("Repository already exists. Use git pull if you need updates.")

Cloning into 'MARL-TSC-SUMO-Group'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 47 (delta 10), reused 38 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 98.41 KiB | 4.69 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
#!unzip traffic_marl_project.zip

# 1. Environment & Dependency Setup
*Installing SUMO, PettingZoo, and configuring system paths.*

In [2]:
!apt-get update && apt-get install -y sumo sumo-tools
!pip install traci pettingzoo sumolib torch torchvision matplotlib

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe a

In [ ]:
#/content/MARL-TSC-SUMO-Group/sumo

In [3]:
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"

In [4]:
import sys
import os
# Point to the root where the unzipped 'env' and 'agents' folders are
sys.path.append('/content/MARL-TSC-SUMO-Group/')

import traci

# 2. Traffic Network Generation
*Using netgenerate and randomTrips to create the 3x3 grid and traffic flows.*

In [6]:
!netgenerate \
--grid \
--grid.number=3 \
--tls.guess true \
--default.lanenumber=3 \
--grid.length=500 \
--default.speed=16.7 \
-o /content/MARL-TSC-SUMO-Group/sumo/network.net.xml

Success.


In [32]:
!python /usr/share/sumo/tools/randomTrips.py \
-n /content/MARL-TSC-SUMO-Group/sumo/network.net.xml \
-r /content/MARL-TSC-SUMO-Group/sumo/routes.rou.xml \
--begin 0 --end 3600 \
--flows 1000 --seed 42

calling /usr/share/sumo/bin/duarouter -n /content/MARL-TSC-SUMO-Group/sumo/network.net.xml -r trips.trips.xml --ignore-errors --begin 0 --end 3600 --no-step-log --no-warnings -o /content/MARL-TSC-SUMO-Group/sumo/routes.rou.xml
Success.


# 3. Multi-Agent Environment Definition
*The custom PettingZoo wrapper for the SUMO simulation.*

In [11]:
import traci
import os

# Diagnostic script to see what SUMO named your intersections
try:
    # Updated path to the repository subfolder
    traci.start(["sumo", "-c", "/content/MARL-TSC-SUMO-Group/sumo/config.sumocfg"])
    tls_ids = traci.trafficlight.getIDList()
    print(f"Detected Traffic Light IDs in SUMO: {tls_ids}")
    traci.close()
except Exception as e:
    print(f"Error during ID check: {e}")
    if "default" in traci.connection._connections: del traci.connection._connections["default"]

 Retrying in 1 seconds
Detected Traffic Light IDs in SUMO: ('A1', 'B0', 'B1', 'B2', 'C1')


# 4. Deep Q-Network Training
*Implementing the DQN Agent, Replay Buffer, and training loop.*

### Training Cell

In [39]:
import sys
import importlib
import env.traffic_env
import utils.metrics
import agents.replay_buffer
importlib.reload(env.traffic_env)
importlib.reload(utils.metrics)
importlib.reload(agents.replay_buffer)

from env.traffic_env import TrafficEnv
from utils.metrics import average_delay
from agents.replay_buffer import ReplayBuffer
from agents.dqn import DQN

import torch
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import traci

def train():
    """
    Main Training Loop: Deep Q-Learning with Experience Replay.

    Theoretical Objective:
    Learn a policy Pi that maps traffic states to signal adjustments to maximize
    long-term discounted reward: G_t = sum(gamma^k * R_{t+k+1}).
    """
    env = TrafficEnv()
    state_size = 7    # [Density_L1, Queue_L1, ExitSpace_L1, Density_L2, Queue_L2, ExitSpace_L2, GlobalMetric]
    action_size = 3   # Actions: 0 (Maintain), 1 (Extend +5s), 2 (Reduce -5s)

    # --- MARL SGAT HYPERPARAMETERS ---
    batch_size = 32     # Number of experiences sampled for each Stochastic Gradient Descent update
    gamma = 0.95        # Discount Factor: Prioritizes immediate vs future rewards (from paper)
    learning_rate = 1e-4
    scaling_factor_q = 1.0 # Used for variable range unification as per SGAT methodology

    # The Q-Network: Function approximator for the Q-value Q(s, a)
    dqn_agent = DQN(state_size, action_size)
    optimizer = optim.Adam(dqn_agent.parameters(), lr=learning_rate)

    # The Replay Buffer: Stores past transitions to break temporal correlations in sequential data
    memory = ReplayBuffer(size=10000)

    num_test_episodes = 5

    for episode in range(num_test_episodes):
        observations = env.reset()
        total_episode_reward = 0.0
        episode_delays = []

        print(f"\n--- Testing Episode {episode + 1}/{num_test_episodes} ---")

        # Simulation duration: 3600 seconds (1 full hour)
        for step in range(3600):
            actions = {}

            # --- 1. ACTION SELECTION (Epsilon-Greedy Strategy) ---
            for agent_id, obs in observations.items():
                obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                with torch.no_grad():
                    q_values = dqn_agent(obs_tensor)

                # 10% chance to explore (random action), 90% to exploit (argmax Q-value)
                if random.random() < 0.1:
                    action = random.randint(0, action_size - 1)
                else:
                    action = q_values.argmax(dim=1).item()
                actions[agent_id] = action

            # --- 2. ENVIRONMENT STEP ---
            # Apply phase duration adjustments to SUMO via PettingZoo wrapper
            next_obs, rewards, terminations, truncations, infos = env.step(actions)

            # Metric Tracking: Using accumulated waiting time for RL feedback
            current_step_delay = average_delay()
            if current_step_delay > 0:
                episode_delays.append(current_step_delay)

            if step % 500 == 0:
                veh_count = traci.vehicle.getIDCount()
                print(f"  Step {step}: Active Vehicles = {veh_count}, Current Delay = {current_step_delay:.2f}s")

            # --- 3. LEARN FROM EXPERIENCE ---
            for agent_id in observations.keys():
                # Store transition tuple: (State, Action, Reward, Next State)
                transition = (observations[agent_id], actions[agent_id], rewards[agent_id], next_obs[agent_id])
                memory.push(transition)

                # Perform SGD update if buffer has sufficient data
                if len(memory) > batch_size:
                    batch = memory.sample(batch_size)
                    # Unzip transitions into separate component batches
                    states, batch_actions, batch_rewards, next_states = zip(*batch)

                    states = torch.tensor(np.array(states), dtype=torch.float32)
                    batch_actions = torch.tensor(batch_actions, dtype=torch.long).unsqueeze(1)
                    batch_rewards = torch.tensor(batch_rewards, dtype=torch.float32).unsqueeze(1)
                    next_states = torch.tensor(np.array(next_states), dtype=torch.float32)

                    # Current Q prediction: Q(s, a)
                    # Multiplied by scaling_factor_q for unification as per the paper
                    current_q = dqn_agent(states).gather(1, batch_actions) * scaling_factor_q

                    # Target Q using Bellman Optimality: Target = Reward + gamma * max[Q(s', a')]
                    # We detach() to stop gradients from flowing into the target calculation
                    max_next_q = dqn_agent(next_states).detach().max(1)[0].unsqueeze(1)
                    expected_q = batch_rewards + (gamma * max_next_q)

                    # Calculate Temporal Difference (TD) Error using Mean Squared Error loss
                    loss = F.mse_loss(current_q, expected_q)

                    # Backpropagation: Adjust neural network weights to minimize TD Error
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            total_episode_reward += sum(rewards.values())
            observations = next_obs

        # End of Episode Reporting
        avg_ep_delay = np.mean(episode_delays) if episode_delays else 0.0
        print(f"  Episode Reward: {total_episode_reward:.2f}, Avg Episode Delay: {avg_ep_delay:.2f}s")

if __name__ == '__main__':
    train()

 Retrying in 1 seconds

--- Testing Episode 1/5 ---
  Step 0: Active Vehicles = 24, Current Delay = 0.00s
  Step 500: Active Vehicles = 351, Current Delay = 50.75s
  Step 1000: Active Vehicles = 24, Current Delay = 0.00s
  Step 1500: Active Vehicles = 374, Current Delay = 48.71s
  Step 2000: Active Vehicles = 24, Current Delay = 0.00s
  Step 2500: Active Vehicles = 388, Current Delay = 46.67s
  Step 3000: Active Vehicles = 24, Current Delay = 0.00s
  Step 3500: Active Vehicles = 364, Current Delay = 48.81s
  Episode Reward: 323772.11, Avg Episode Delay: 35.41s
 Retrying in 1 seconds

--- Testing Episode 2/5 ---
  Step 0: Active Vehicles = 24, Current Delay = 0.00s
  Step 500: Active Vehicles = 351, Current Delay = 50.75s
  Step 1000: Active Vehicles = 24, Current Delay = 0.00s
  Step 1500: Active Vehicles = 367, Current Delay = 46.97s
  Step 2000: Active Vehicles = 24, Current Delay = 0.00s
  Step 2500: Active Vehicles = 381, Current Delay = 45.41s
  Step 3000: Active Vehicles = 24, Cu

# 5. Persistence & GitHub Sync
*Backing up files to Google Drive and pushing to the repository.*

In [29]:
!cp -r "/content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC/." /content/MARL-TSC-SUMO-Group/


In [30]:
%cd /content/MARL-TSC-SUMO-Group/
!git config --global user.email "zakfayle@gmail.com"
!git config --global user.name "Isaac Fayle-Waters"
!git add .
!git commit -m "Update from Colab with improved TrafficEnv metrics"


/content/MARL-TSC-SUMO-Group
[main 6299179] Update from Colab with improved TrafficEnv metrics
 16 files changed, 33209 insertions(+)
 create mode 100644 agents/__pycache__/dqn.cpython-312.pyc
 create mode 100644 agents/__pycache__/replay_buffer.cpython-312.pyc
 create mode 100644 agents/dqn.py
 create mode 100644 agents/replay_buffer.py
 create mode 100644 agents/train.py
 create mode 100644 env/__pycache__/traffic_env.cpython-312.pyc
 create mode 100644 env/observation.py
 create mode 100644 env/traffic_env.py
 create mode 100644 extracted_configuration.json
 create mode 100644 sumo/config.sumocfg
 create mode 100644 sumo/network.net.xml
 create mode 100644 sumo/routes.rou.alt.xml
 create mode 100644 sumo/routes.rou.xml
 create mode 100644 traffic_marl_project.zip
 create mode 100644 utils/__pycache__/metrics.cpython-312.pyc
 create mode 100644 utils/metrics.py


In [ ]:
!git push origin main

In [38]:
from google.colab import userdata
import os

# 1. Get the token from Colab Secrets
token = userdata.get('GH_TOKEN')
repo_name = "IsaacFayle-Waters/MARL-TSC-SUMO-Group"

# 2. Configure and Push from the repo root
%cd /content/MARL-TSC-SUMO-Group/

# Ensure identity is known for this commit
!git config user.email "zakfayle@gmail.com"
!git config user.name "Isaac Fayle-Waters"

# Update local repo with latest library files from /content before pushing
!cp -r /content/MARL-TSC-SUMO-Group/env /content/MARL-TSC-SUMO-Group/agents /content/MARL-TSC-SUMO-Group/utils .

# Try to find and copy the current notebook file (.ipynb) to the repo root
!find /content -maxdepth 1 -name "*.ipynb" -exec cp {} . \;

!git add .
!git commit -m "Update: 1-hour simulation, calibrated hyperparameters, and usage instructions"
!git remote set-url origin https://{token}@github.com/{repo_name}.git
!git push origin main

/content/MARL-TSC-SUMO-Group
cp: '/content/MARL-TSC-SUMO-Group/env' and './env' are the same file
cp: '/content/MARL-TSC-SUMO-Group/agents' and './agents' are the same file
cp: '/content/MARL-TSC-SUMO-Group/utils' and './utils' are the same file
[main d29b9ab] Update: 1-hour simulation, calibrated hyperparameters, and usage instructions
 4 files changed, 1019 insertions(+), 6 deletions(-)
 create mode 100644 trips.trips.xml
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 7.85 KiB | 7.85 MiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/IsaacFayle-Waters/MARL-TSC-SUMO-Group.git
   3765108..d29b9ab  main -> main


# Writefiles for Faster updating and reference

In [37]:
%%writefile /content/agents/dqn.py
"""
Deep Q-Network (DQN) Implementation.
This model approximates the Q-value function for traffic signal control decisions.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        """
        Initialize the Neural Network.
        Args:
            state_size (int): Number of input features (Density/Queuing metrics).
            action_size (int): Number of possible traffic light phases.
        """
        super(DQN, self).__init__()
        # Standard multi-layer perceptron architecture
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):
        """
        Forward pass to predict Q-values for each action.
        """
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

Overwriting /content/agents/dqn.py


In [38]:
%%writefile /content/agents/replay_buffer.py
"""
Replay Buffer for Experience Replay.
Stores past transitions to break temporal correlation during training.
"""
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, size=10000):
        """
        Initialize a circular buffer using a deque.
        """
        self.buffer = deque(maxlen=size)

    def push(self, transition):
        """
        Add a new experience (s, a, r, s') to the buffer.
        """
        self.buffer.append(transition)

    def sample(self, batch_size):
        """
        Randomly sample a batch of experiences for stochastic gradient descent.
        """
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

Overwriting /content/agents/replay_buffer.py


In [25]:
%%writefile /content/MARL-TSC-SUMO-Group/utils/metrics.py
"""
Utility functions for calculating traffic performance metrics via TraCI.
Optimized for Reinforcement Learning feedback.
"""
import traci

def average_delay():
    """
    Calculates the average accumulated waiting time across all vehicles currently in the simulation.
    Using accumulatedWaitingTime is more robust for RL than getWaitingTime.
    """
    veh_ids = traci.vehicle.getIDList()
    if len(veh_ids) == 0:
        return 0.0

    total_accumulated_delay = 0.0
    for v in veh_ids:
        # getAccumulatedWaitingTime tracks total time spent at speed < 0.1m/s
        total_accumulated_delay += traci.vehicle.getAccumulatedWaitingTime(v)

    return total_accumulated_delay / len(veh_ids)

Overwriting /content/MARL-TSC-SUMO-Group/utils/metrics.py


### 📝 TODO List for Experiment Alignment

- [x] **Action Space**: Transition from discrete phase switching to 'Phase Duration Adjustment' (delta_t adjustment).
- [x] **State Space**: Expand observations to include 'Remaining Exit Space' and 'Global Network Metrics'.
- [ ] **Network Scale**: Increase grid from 3x3 (9 agents) to 5x5 (25 agents) to match the paper's scenario.
- [ ] **Reward Tuning**: Implement the 'Heterogeneous Correlation Index' and calibrate weights for delay vs. throughput.
- [x] **Simulation Length**: Increase episode duration to 3600 seconds (1 full hour).
- [x] **Hyperparameters**: Verify 'Scaling Factor Q' and 'Discount Factor (0.95)' across all training scripts.

In [28]:
with open('/content/MARL-TSC-SUMO-Group/utils/metrics.py', 'r') as f:
    print(f.read())

"""
Utility functions for calculating traffic performance metrics via TraCI.
Optimized for Reinforcement Learning feedback.
"""
import traci

def average_delay():
    """
    Calculates the average accumulated waiting time across all vehicles currently in the simulation.
    Using accumulatedWaitingTime is more robust for RL than getWaitingTime.
    """
    veh_ids = traci.vehicle.getIDList()
    if len(veh_ids) == 0:
        return 0.0

    total_accumulated_delay = 0.0
    for v in veh_ids:
        # getAccumulatedWaitingTime tracks total time spent at speed < 0.1m/s
        total_accumulated_delay += traci.vehicle.getAccumulatedWaitingTime(v)

    return total_accumulated_delay / len(veh_ids)



In [20]:
%%writefile /content/MARL-TSC-SUMO-Group/env/observation.py
"""
Observation logic for MARL SGAT.
Calculates local lane metrics and global network inflow/outflow.
"""
import traci
import numpy as np

class MARLObservation:
    def __init__(self, sumo_id):
        self.sumo_id = sumo_id
        self.lanes = traci.trafficlight.getControlledLanes(sumo_id)
        self.unique_lanes = list(dict.fromkeys(self.lanes))

    def get(self):
        """
        Returns a 7-dimensional observation vector:
        [Density_L1, Queue_L1, ExitSpace_L1, Density_L2, Queue_L2, ExitSpace_L2, GlobalMetric]
        """
        obs = []
        # 1. Local Metrics for the first 2 controlled lanes
        for lane in self.unique_lanes[:2]:
            length = traci.lane.getLength(lane)
            veh_num = traci.lane.getLastStepVehicleNumber(lane)

            # Density
            obs.append(veh_num / length)
            # Queuing Density
            obs.append(traci.lane.getLastStepHaltingNumber(lane) / length)
            # Remaining Exit Space
            max_cap = length / 7.5
            obs.append(max(0, 1 - (veh_num / max_cap)))

        # 2. Global Network Metric (Approximated by total expected vehicles)
        n_total = traci.simulation.getMinExpectedNumber()
        obs.append(n_total / 100.0)

        # Padding to ensure fixed size of 7
        while len(obs) < 7: obs.append(0.0)
        return np.array(obs, dtype=np.float32)

Overwriting /content/MARL-TSC-SUMO-Group/env/observation.py


In [21]:
%%writefile /content/MARL-TSC-SUMO-Group/env/traffic_env.py
from pettingzoo import ParallelEnv
import traci
import traci.connection
import numpy as np
from env.observation import MARLObservation

class TrafficEnv(ParallelEnv):
    metadata = {"name": "traffic_marl_env"}

    def __init__(self):
        self.sumo_ids = ['A1', 'B0', 'B1', 'B2', 'C1']
        self.agents = [f"agent_{i}" for i in range(len(self.sumo_ids))]
        self.agent_to_sumo = {f"agent_{i}": sid for i, sid in enumerate(self.sumo_ids)}

        self.min_green = 10
        self.max_green = 60
        self.delta_t = 5

        self.last_waiting_times = {agent: 0 for agent in self.agents}
        self.current_phase_timer = {agent: 0 for agent in self.agents}
        # Initialize observation objects for each agent
        self.observations = {agent: MARLObservation(self.agent_to_sumo[agent]) for agent in self.agents}

    def reset(self, seed=None, options=None):
        try:
            if "default" in traci.connection._connections: traci.close()
        except Exception: pass
        finally:
            if "default" in traci.connection._connections: del traci.connection._connections["default"]

        traci.start(["sumo", "-c", "/content/MARL-TSC-SUMO-Group/sumo/config.sumocfg"])

        self.last_waiting_times = {agent: 0 for agent in self.agents}
        self.current_phase_timer = {agent: 0 for agent in self.agents}
        return {agent: self.observations[agent].get() for agent in self.agents}

    def step(self, actions):
        for agent, action in actions.items():
            self._apply_action(agent, action)

        traci.simulationStep()
        for agent in self.agents:
            self.current_phase_timer[agent] += 1

        obs = {agent: self.observations[agent].get() for agent in self.agents}
        rewards = {agent: self._compute_reward(agent) for agent in self.agents}
        return obs, rewards, {a: False for a in self.agents}, {a: False for a in self.agents}, {a: {} for a in self.agents}

    def _apply_action(self, agent, action):
        sumo_id = self.agent_to_sumo[agent]
        current_phase = traci.trafficlight.getPhase(sumo_id)

        if current_phase % 2 != 0:
            if self.current_phase_timer[agent] >= 3:
                traci.trafficlight.setPhase(sumo_id, (current_phase + 1) % 4)
                self.current_phase_timer[agent] = 0
            return

        if action == 1:
            new_duration = min(self.max_green, traci.trafficlight.getPhaseDuration(sumo_id) + self.delta_t)
            traci.trafficlight.setPhaseDuration(sumo_id, new_duration)
        elif action == 2:
            new_duration = max(self.min_green, traci.trafficlight.getPhaseDuration(sumo_id) - self.delta_t)
            traci.trafficlight.setPhaseDuration(sumo_id, new_duration)

        if self.current_phase_timer[agent] >= traci.trafficlight.getPhaseDuration(sumo_id):
            traci.trafficlight.setPhase(sumo_id, (current_phase + 1) % 4)
            self.current_phase_timer[agent] = 0

    def _compute_reward(self, agent):
        sumo_id = self.agent_to_sumo[agent]
        lanes = traci.trafficlight.getControlledLanes(sumo_id)
        current_waiting_time = sum([traci.lane.getWaitingTime(l) for l in lanes])
        throughput = sum([traci.lane.getLastStepVehicleNumber(l) for l in lanes])
        delay_component = (self.last_waiting_times[agent] - current_waiting_time) / 100.0
        reward = delay_component + (throughput * 0.1)
        self.last_waiting_times[agent] = current_waiting_time
        return float(reward)

Overwriting /content/MARL-TSC-SUMO-Group/env/traffic_env.py


In [37]:
readme_content = """# MARL-TSC-SUMO-Group

## Project Purpose
This project is an experiment reproduction of the **'Multi-agent Deep Reinforcement Learning collaborative Traffic Signal Control method considering intersection heterogeneity'** (MARL SGAT).

## Objective
The goal is to maximize long-term cumulative reward (minimizing delay and maximizing throughput) across a road network using a 5x5 grid of 25 intersections controlled by decentralized DQN agents.

## Usage Instructions
1. **Environment Setup**: Open the `experiment_reprod_bie_et_al.ipynb` notebook in Google Colab.
2. **Initialization**: Run the first cell to clone this repository and install dependencies (SUMO, PettingZoo, etc.).
3. **Network Generation**: Execute the traffic network generation cells to build the grid and traffic flows.
4. **Training**: Run the training loop cell to begin the DQN optimization. The notebook is pre-configured with the MARL SGAT hyperparameters.
5. **Evaluation**: Use the provided evaluation cells to visualize agent actions and calculate average vehicle delay.

## Current Status
- Environment: PettingZoo-compatible wrapper for SUMO with Phase Duration Adjustment.
- Agent: Deep Q-Network (DQN) with Replay Buffer and SGAT Hyperparameters.
- Metrics: Average vehicle delay, density, queuing density, and throughput proxy.
"""
with open('/content/MARL-TSC-SUMO-Group/README.md', 'w') as f:
    f.write(readme_content)
print('README updated with usage instructions.')

README updated with usage instructions.


In [15]:
!pip install pypdf
from pypdf import PdfReader

reader = PdfReader('/content/bie_et_al_marl_spaciotemp.pdf')
text = ''
for page in reader.pages:
    text += page.extract_text() + "\n"

# Broader search with case insensitivity
keywords = ['exit space', 'global', 'n_in', 'n_out', 'observation', 'state space']
lines = text.split('\n')

for i, line in enumerate(lines):
    if any(kw.lower() in line.lower() for kw in keywords):
        # Print the line and the next few lines for context
        print(f"--- Found at line {i} ---")
        print('\n'.join(lines[i:i+5]))
        print("-"*30)

--- Found at line 35 ---
On these bases, double dual networks are introduced to convert the individual-global-max
constraint of action selection into a value range constraint of the action advantage function,
thereby facilitating model learning. Simulation results showcase the superior performance of the
MARL_SGAT algorithm when compared to seven baseline algorithms. Specifically, the algorithm
manifests in reduced average vehicle delays, decreased number of stops, and elevated travel
------------------------------
--- Found at line 56 ---
these, adaptive control stands out as it can achieve and maintain a global performance level for the control system by accommodating
fluctuations in traffic parameters over time. It serves as a prevalent approach to address signal control challenges.
In recent years, the integration of machine learning methods into TSC has become increasingly prevalent with the advent of
artificial intelligence (Qin et al., 2022; Qu et al., 2023). Reinforcement learn

In [18]:
%%writefile /content/MARL-TSC-SUMO-Group/env/traffic_env.py
"""
TrafficEnv: Updated with MARL SGAT Observation Space and Phase Duration Adjustment Action Space.

ACTION SPACE LOGIC (Phase Duration Adjustment):
- Action 0: Maintain current green time.
- Action 1: Extend current green phase by delta_t (e.g., +5s).
- Action 2: Reduce current green phase by delta_t (e.g., -5s).

This version manages internal timers to ensure minimum/maximum green times and handles
transitions to yellow phases automatically.
"""
from pettingzoo import ParallelEnv
import traci
import traci.connection
import numpy as np

class TrafficEnv(ParallelEnv):
    metadata = {"name": "traffic_marl_env"}

    def __init__(self):
        self.sumo_ids = ['A1', 'B0', 'B1', 'B2', 'C1']
        self.agents = [f"agent_{i}" for i in range(len(self.sumo_ids))]
        self.agent_to_sumo = {f"agent_{i}": sid for i, sid in enumerate(self.sumo_ids)}

        # Simulation constants
        self.min_green = 10
        self.max_green = 60
        self.delta_t = 5

        # Internal state tracking
        self.last_waiting_times = {agent: 0 for agent in self.agents}
        self.current_phase_timer = {agent: 0 for agent in self.agents}
        self.is_yellow = {agent: False for agent in self.agents}

    def reset(self, seed=None, options=None):
        """Resets simulation and initializes timers."""
        try:
            if "default" in traci.connection._connections: traci.close()
        except Exception: pass
        finally:
            if "default" in traci.connection._connections: del traci.connection._connections["default"]

        traci.start(["sumo", "-c", "/content/MARL-TSC-SUMO-Group/sumo/config.sumocfg"])

        self.last_waiting_times = {agent: 0 for agent in self.agents}
        self.current_phase_timer = {agent: 0 for agent in self.agents}
        self.is_yellow = {agent: False for agent in self.agents}

        return {agent: self._get_obs(agent) for agent in self.agents}

    def step(self, actions):
        """Executes Phase Duration Adjustments and steps the simulation."""
        for agent, action in actions.items():
            self._apply_action(agent, action)

        # Increment internal timers per simulation step (assumed 1s per step)
        traci.simulationStep()
        for agent in self.agents:
            self.current_phase_timer[agent] += 1

        observations = {agent: self._get_obs(agent) for agent in self.agents}
        rewards = {agent: self._compute_reward(agent) for agent in self.agents}
        return observations, rewards, {a: False for a in self.agents}, {a: False for a in self.agents}, {a: {} for a in self.agents}

    def _get_obs(self, agent):
        """State: [Density x2, Queue x2, ExitSpace x2, GlobalMetric]. Total=7"""
        sumo_id = self.agent_to_sumo[agent]
        lanes = traci.trafficlight.getControlledLanes(sumo_id)
        unique_lanes = list(dict.fromkeys(lanes))

        obs = []
        for lane in unique_lanes[:2]:
            length = traci.lane.getLength(lane)
            veh_num = traci.lane.getLastStepVehicleNumber(lane)
            obs.append(veh_num / length) # Density
            obs.append(traci.lane.getLastStepHaltingNumber(lane) / length) # Queue Density
            max_cap = length / 7.5
            obs.append(max(0, 1 - (veh_num / max_cap))) # Remaining Exit Space

        n_total = traci.simulation.getMinExpectedNumber()
        obs.append(n_total / 100.0) # Global metric

        while len(obs) < 7: obs.append(0.0)
        return np.array(obs, dtype=np.float32)

    def _apply_action(self, agent, action):
        """
        Logic for Phase Duration Adjustment:
        0: Keep, 1: +5s, 2: -5s
        Ensures transitions to yellow if green expires.
        """
        sumo_id = self.agent_to_sumo[agent]
        current_phase = traci.trafficlight.getPhase(sumo_id)

        # If in yellow phase (odd indices in standard grids), just wait for it to end
        if current_phase % 2 != 0:
            if self.current_phase_timer[agent] >= 3: # Standard 3s yellow
                traci.trafficlight.setPhase(sumo_id, (current_phase + 1) % 4)
                self.current_phase_timer[agent] = 0
            return

        # Adjust Green Time based on action
        if action == 1: # Extend
            new_duration = min(self.max_green, traci.trafficlight.getPhaseDuration(sumo_id) + self.delta_t)
            traci.trafficlight.setPhaseDuration(sumo_id, new_duration)
        elif action == 2: # Reduce
            new_duration = max(self.min_green, traci.trafficlight.getPhaseDuration(sumo_id) - self.delta_t)
            traci.trafficlight.setPhaseDuration(sumo_id, new_duration)

        # Check if the green time has expired locally
        if self.current_phase_timer[agent] >= traci.trafficlight.getPhaseDuration(sumo_id):
            traci.trafficlight.setPhase(sumo_id, (current_phase + 1) % 4)
            self.current_phase_timer[agent] = 0

    def _compute_reward(self, agent):
        sumo_id = self.agent_to_sumo[agent]
        lanes = traci.trafficlight.getControlledLanes(sumo_id)
        current_waiting_time = sum([traci.lane.getWaitingTime(l) for l in lanes])
        throughput = sum([traci.lane.getLastStepVehicleNumber(l) for l in lanes])
        delay_component = (self.last_waiting_times[agent] - current_waiting_time) / 100.0
        reward = delay_component + (throughput * 0.1)
        self.last_waiting_times[agent] = current_waiting_time
        return float(reward)

Overwriting /content/MARL-TSC-SUMO-Group/env/traffic_env.py


In [46]:
import torch
from env.traffic_env import TrafficEnv
from agents.dqn import DQN
import traci

def observe_policy_behavior(steps=20):
    """
    Runs a short simulation to print the real-time decisions of the agent.
    Useful for verifying that the agent isn't 'stuck' in a specific phase.
    """
    env = TrafficEnv()
    state_size = 6
    action_size = 4

    # Initialize and set to evaluation mode (no weight updates)
    agent = DQN(state_size, action_size)
    agent.eval()

    obs = env.reset()
    print(f"{'Step':<5} | {'Agent':<8} | {'Action':<7} | {'SUMO Phase State'}")
    print("-" * 45)

    for s in range(steps):
        actions = {}
        for agent_id, o in obs.items():
            # Convert observation to tensor for the Neural Network
            o_tensor = torch.tensor(o, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                # Get the action with the highest predicted Q-value
                action = agent(o_tensor).argmax(dim=1).item()
            actions[agent_id] = action

            # Fetch the actual 'G/r/y' string from the SUMO simulation core
            sumo_id = env.agent_to_sumo[agent_id]
            phase_state = traci.trafficlight.getRedYellowGreenState(sumo_id)

            # Print periodic updates to track behavior
            if s % 5 == 0:
                print(f"{s:<5} | {agent_id:<8} | {action:<7} | {phase_state}")

        # Step the environment with the chosen actions
        obs, _, _, _, _ = env.step(actions)

    traci.close()

if __name__ == '__main__':
    observe_policy_behavior()

 Retrying in 1 seconds
Step  | Agent    | Action  | SUMO Phase State
---------------------------------------------
0     | agent_0  | 2       | GGGgggrrrrrGGGGg
0     | agent_1  | 2       | rrrrrGGGGgGGGggg
0     | agent_2  | 2       | GGGGggrrrrrrGGGGggrrrrrr
0     | agent_3  | 2       | GGGgggrrrrrGGGGg
0     | agent_4  | 2       | GGGGgGGGgggrrrrr
5     | agent_0  | 2       | rrrrrrGGGGgGrrrr
5     | agent_1  | 2       | GGGGgGrrrrrrrrrr
5     | agent_2  | 2       | rrrrrrGGGGggrrrrrrGGGGgg
5     | agent_3  | 2       | rrrrrrGGGGgGrrrr
5     | agent_4  | 2       | GrrrrrrrrrrGGGGg
10    | agent_0  | 2       | rrrrrrGGGGgGrrrr
10    | agent_1  | 2       | GGGGgGrrrrrrrrrr
10    | agent_2  | 2       | rrrrrrGGGGggrrrrrrGGGGgg
10    | agent_3  | 2       | rrrrrrGGGGgGrrrr
10    | agent_4  | 2       | GrrrrrrrrrrGGGGg
15    | agent_0  | 2       | rrrrrrGGGGgGrrrr
15    | agent_1  | 2       | GGGGgGrrrrrrrrrr
15    | agent_2  | 2       | rrrrrrGGGGggrrrrrrGGGGgg
15    | agent_3  | 2     

In [26]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Updated path based on your specific Uni directory
project_path = '/content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC'

if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f'Created directory: {project_path}')

# Execute this to sync your current work to Google Drive
!cp -r /content/env /content/sumo /content/agents /content/utils /content/extracted_configuration.json "{project_path}"
print(f'Files successfully backed up to {project_path}')

Mounted at /content/drive
Files successfully backed up to /content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC


### GitHub Persistence
To connect this project to GitHub from Colab:

1. **Generate a Personal Access Token (PAT)** on GitHub.
2. **Initialize Git** in your project folder (e.g., in `/content/drive/MyDrive/traffic_marl_project`).
3. **Configure your identity** and push using the PAT as the password.

```python
# Example for pushing
!git config --global user.email "your_email@example.com"
!git config --global user.name "Your Name"
!git remote add origin https://<TOKEN>@github.com/<USERNAME>/<REPO>.git
!git add .
!git commit -m "Sync from Colab"
!git push -u origin main
```

In [14]:
import torch
from env.traffic_env import TrafficEnv
from agents.dqn import DQN
import numpy as np

def evaluate(model_path=None):
    env = TrafficEnv()
    state_size = 6
    action_size = 4

    # Initialize agent
    eval_agent = DQN(state_size, action_size)

    # If you saved a model, load it here:
    # eval_agent.load_state_dict(torch.load(model_path))
    eval_agent.eval() # Set to evaluation mode

    observations = env.reset()
    total_reward = 0
    steps = 0

    print("Starting Evaluation...")

    for step in range(300):
        actions = {}
        for agent_id, obs in observations.items():
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                q_values = eval_agent(obs_tensor)
            # Take the best action (greedy)
            action = q_values.argmax(dim=1).item()
            actions[agent_id] = action

        next_obs, rewards, terms, truncs, infos = env.step(actions)
        total_reward += sum(rewards.values())
        observations = next_obs
        steps += 1

    print(f"Evaluation Finished.")
    print(f"Total Reward: {total_reward:.2f}")
    print(f"Steps: {steps}")

if __name__ == '__main__':
    evaluate()

 Retrying in 1 seconds
Starting Evaluation...
Evaluation Finished.
Total Reward: 0.00
Steps: 300


#### LLM USAGE:
##### Main Extraction and Files
From ChatGPT --> https://chatgpt.com/share/69b57d24-5940-8011-a264-6d93f952b33b
#### Assistance From Colab's Inbuilt Gemini Instances
- Gemini 2.5 Flash
- Gemini 3 Flash


Editing and ensuring reproducable environment.
